In [1]:
import pandas as pd
import re
import emoji
import html
from tqdm import tqdm

tqdm.pandas()
print("Library berhasil dimuat")

Library berhasil dimuat


In [2]:
df = pd.read_csv("rawData/dataX.csv")
df = df[['created_at', 'user_id_str', 'full_text']]
df['full_text'] = df['full_text'].fillna('')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3714 entries, 0 to 3713
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   created_at   3714 non-null   object
 1   user_id_str  3714 non-null   int64 
 2   full_text    3714 non-null   object
dtypes: int64(1), object(2)
memory usage: 87.2+ KB


In [3]:
initial_count = len(df)
df.drop_duplicates(subset=['full_text'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Data dimuat: {len(df)} tweet (dihapus {initial_count - len(df)} duplikat)")
print(f"Kolom: {df.columns.tolist()}")

Data dimuat: 1882 tweet (dihapus 1832 duplikat)
Kolom: ['created_at', 'user_id_str', 'full_text']


In [4]:
def clean_tweet_bertopic(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = emoji.demojize(text, language='en')
    text = re.sub(r':[\w_]+:', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r' \1', text)
    text = re.sub(r'^RT[\s]+', '', text)
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def case_folding(text):
    if not isinstance(text, str):
        return ""
    return text.lower()

print("Fungsi cleaning dan case folding siap")

Fungsi cleaning dan case folding siap


In [5]:
df['data_cleaning'] = df['full_text'].progress_apply(clean_tweet_bertopic)
df['case_folding']  = df['data_cleaning'].progress_apply(case_folding)

print("Cleaning dan case folding selesai")

100%|██████████| 1882/1882 [00:00<00:00, 943430.16it/s]

Cleaning dan case folding selesai


In [6]:
kamus_alay = pd.read_csv(
    "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
)

slang_dict = (
    kamus_alay.drop_duplicates(subset=['slang'])
    .set_index('slang')['formal']
    .to_dict()
)

manual_dict = {
    'yg': 'yang', 'dg': 'dengan', 'dgn': 'dengan',
    'tdk': 'tidak', 'krn': 'karena', 'sdh': 'sudah',
    'udh': 'sudah', 'bs': 'bisa', 'utk': 'untuk',
    'bgt': 'sangat', 'banget': 'sangat', 'tp': 'tapi',
    'skrng': 'sekarang', 'skrg': 'sekarang', 'tau': 'tahu',
    'gk': 'tidak', 'gak': 'tidak', 'ga': 'tidak',
    'kpd': 'kepada', 'tsb': 'tersebut', 'ttg': 'tentang',
    'thd': 'terhadap', 'dr': 'dari', 'pd': 'pada',
    'dlm': 'dalam', 'jd': 'jadi',
    # Partikel penegas → hapus
    'deh': '', 'sih': '', 'dong': '', 'kok': '',
    'loh': '', 'lho': '', 'nah': '', 'nih': '',
    'tuh': '',
}

slang_dict.update(manual_dict)

print(f"Kamus alay dimuat : {len(kamus_alay)} entri")
print(f"Setelah merge     : {len(slang_dict)} entri unik")

Kamus alay dimuat : 15006 entri
Setelah merge     : 4343 entri unik


In [7]:
def normalize_with_kamus(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    normalized = []
    for word in words:
        replacement = slang_dict.get(word)
        if replacement is None:
            normalized.append(word)
        elif replacement:
            normalized.append(replacement)
        # pengganti kosong (partikel) → buang
    return ' '.join(normalized)

df['normalization'] = df['case_folding'].progress_apply(normalize_with_kamus)
print("Normalisasi dengan kamus alay selesai")

100%|██████████| 1882/1882 [00:00<00:00, 188352.86it/s]

Normalisasi dengan kamus alay selesai


In [8]:
print("=" * 60)
print("LAPORAN KUALITAS PREPROCESSING — DATA X (BERTopic)")
print("=" * 60)

df['orig_length']  = df['full_text'].str.len()
df['final_length'] = df['normalization'].str.len()
df['orig_words']   = df['full_text'].apply(lambda x: len(str(x).split()))
df['final_words']  = df['normalization'].apply(lambda x: len(str(x).split()))

avg_orig    = df['orig_length'].mean()
avg_final   = df['final_length'].mean()
avg_orig_w  = df['orig_words'].mean()
avg_final_w = df['final_words'].mean()

print(f"\nStatistik panjang teks:")
print(f"  Rata-rata panjang asli  : {avg_orig:.1f} karakter")
print(f"  Rata-rata panjang akhir : {avg_final:.1f} karakter")
print(f"  Reduksi                 : {((avg_orig - avg_final) / avg_orig * 100):.1f}%")

print(f"\nStatistik jumlah kata:")
print(f"  Rata-rata kata asli  : {avg_orig_w:.1f}")
print(f"  Rata-rata kata akhir : {avg_final_w:.1f}")
print(f"  Reduksi              : {((avg_orig_w - avg_final_w) / avg_orig_w * 100):.1f}%")

print(f"\nDistribusi panjang teks akhir:")
print(f"  Terpendek : {df['final_length'].min()} karakter")
print(f"  Terpanjang: {df['final_length'].max()} karakter")
print(f"  Median    : {df['final_length'].median():.1f} karakter")

LAPORAN KUALITAS PREPROCESSING — DATA X (BERTopic)

Statistik panjang teks:
  Rata-rata panjang asli  : 167.5 karakter
  Rata-rata panjang akhir : 144.7 karakter
  Reduksi                 : 13.7%

Statistik jumlah kata:
  Rata-rata kata asli  : 23.9
  Rata-rata kata akhir : 22.0
  Reduksi              : 8.0%

Distribusi panjang teks akhir:
  Terpendek : 10 karakter
  Terpanjang: 303 karakter
  Median    : 140.0 karakter


In [9]:
bertopic_data = df[[
    'created_at', 'user_id_str', 'full_text',
    'data_cleaning', 'case_folding', 'normalization',
]].copy()

bertopic_data['preprocessing_stats'] = bertopic_data.apply(
    lambda row: f"orig:{len(row['full_text'])} -> final:{len(row['normalization'])}",
    axis=1
)

output_file = "bertopic_pisah_data/x_bertopic_without_stop_word.csv"
bertopic_data.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data diekspor ke  : {output_file}")
print(f"Jumlah tweet siap : {len(bertopic_data)}")
print(f"Kolom             : {bertopic_data.columns.tolist()}")

Data diekspor ke  : bertopic_pisah_data/x_bertopic_without_stop_word.csv
Jumlah tweet siap : 1882
Kolom             : ['created_at', 'user_id_str', 'full_text', 'data_cleaning', 'case_folding', 'normalization', 'preprocessing_stats']
